In [110]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE
from collections import Counter

**Load Data**

In [111]:

df_original = pd.read_csv('Children Recode_final.csv')
df_original['Malnurished'] = df_original[['Underweight', 'Stunting', 'Wasting']].max(axis=1)
df = df_original.drop(['Underweight', 'Stunting', 'Wasting'], axis = 1)
df.head()
df.shape


(2239, 35)

**Train-test Split**


In [112]:
X = df.drop(columns=['Malnurished'])
y = df['Malnurished']

# Train-test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state= 12)


**Standard Scaler**


In [113]:
columns_to_scale = ['Child_age', 'Age_first_sex', 'BMI', 'Mother_age_current', 'Mother_age_at_first_birth']
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[columns_to_scale] = scaler.fit_transform(X_train[columns_to_scale])
X_test_scaled[columns_to_scale] = scaler.transform(X_test[columns_to_scale])
X_train_scaled.head()

,Child_age,Mother_education,Age_first_sex,Pregnancy_terminated,Wealth_index,Place_residence,BMI,Children_under5,Total_children_ever_born,Child_sex,...,Religion_5,Ethnicity_2,Ethnicity_3,Ethnicity_4,Ethnicity_5,Ethnicity_6,Ethnicity_7,Ethnicity_8,Ethnicity_9,Ethnicity_10
475,-0.014124,2,0.229651,1,4,2,-0.063844,2,1,2,...,0,0,0,0,0,0,0,1,0,0
1573,0.909059,0,1.157279,1,1,2,0.272522,2,2,2,...,0,1,0,0,0,0,0,0,0,0
2108,-1.745092,2,0.229651,0,3,1,-0.853012,0,1,2,...,0,1,0,0,0,0,0,0,0,0
714,-0.591113,1,1.157279,0,2,1,-0.586506,6,2,2,...,0,0,0,1,0,0,0,0,0,0
1412,-1.398898,2,-0.079558,0,4,1,-0.951335,2,1,1,...,0,0,0,0,0,0,0,1,0,0


**Balancing using SMOTE**

In [114]:
sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train_scaled, y_train)
print('Before SMOTE: ', Counter(y_train))
print('After SMOTE: ', Counter(y_train_sm))

Before SMOTE:  Counter({0: 1110, 1: 569})
After SMOTE:  Counter({0: 1110, 1: 1110})


In [115]:
tree = DecisionTreeClassifier(
    class_weight='balanced',
    max_depth=5,
    min_samples_leaf=10
)
tree.fit(X_train_sm, y_train_sm)

DecisionTreeClassifier(class_weight='balanced', max_depth=5,
                       min_samples_leaf=10)

In [116]:
y_pred = tree.predict(X_test_scaled)
print(accuracy_score(y_test, y_pred))
pd.crosstab(y_test, y_pred)

0.5517857142857143


col_0,0,1
Malnurished,,
0,213,158
1,93,96


**Model's Performance**

In [117]:
# Classification Report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.70      0.57      0.63       371
           1       0.38      0.51      0.43       189

    accuracy                           0.55       560
   macro avg       0.54      0.54      0.53       560
weighted avg       0.59      0.55      0.56       560

